In [2]:
import pandas as pd

# Load your student and school datasets
students = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_Student(Seat)')
high_schools = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_High_School')
survey = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='SurveyData_Student(Seat)')

# Filter and clean student data
students = students[students["Participation Status"] == "Participating"]
students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)

# Merge student data with survey and school info
merged = students.merge(survey[["Seat Id", "NPS"]], on="Seat Id", how="left")
merged = merged.merge(high_schools, left_on="School Id", right_on="School Id", how="left")

# Updated get_region function using full state names
def get_region(state):
    northeast = ["New York", "New Jersey", "Massachusetts", "Pennsylvania", "Connecticut",
                 "Rhode Island", "Vermont", "New Hampshire", "Maine"]
    midwest = ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin",
               "Minnesota", "Iowa", "Missouri", "Kansas", "Nebraska", "South Dakota", "North Dakota"]
    south = ["Texas", "Florida", "Georgia", "North Carolina", "South Carolina", "Virginia",
             "Alabama", "Mississippi", "Kentucky", "Tennessee", "Oklahoma", "Arkansas",
             "Louisiana", "West Virginia", "Delaware", "Maryland", "District of Columbia"]
    west = ["California", "Arizona", "Colorado", "Washington", "Oregon", "Nevada",
            "Utah", "New Mexico", "Hawaii", "Alaska", "Idaho", "Montana", "Wyoming"]

    if state in northeast:
        return "Northeast"
    elif state in midwest:
        return "Midwest"
    elif state in south:
        return "South"
    elif state in west:
        return "West"
    else:
        return "Other"

merged["Region"] = merged["State"].apply(get_region)

# Group by school demographics
grouped = merged.groupby(["City", "State", "Locale", "Time Zone", "Region"]).agg(
    Avg_Pass_Rate=("Passed_Flag", "mean"),
    Avg_NPS=("NPS", "mean"),
    Students=("Seat Id", "count")
).reset_index()

# Filter to larger sample sizes if needed
grouped = grouped[grouped["Students"] >= 30]

# View results
grouped.sort_values(by="Avg_Pass_Rate", ascending=False).head(10)


<ipython-input-2-559a9fa4c241>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)


,City,State,Locale,Time Zone,Region,Avg_Pass_Rate,Avg_NPS,Students
75,Kenansville,North Carolina,Rural,Eastern Daylight Time (EDT),South,1.000000,9.05,48
92,Lynn,Massachusetts,Suburb,Eastern Daylight Time (EDT),Northeast,1.000000,8.396552,94
43,Elmont,New York,Suburb,Eastern Daylight Time (EDT),Northeast,0.996094,8.924528,256
79,Lake Balboa,California,City,Pacific Daylight Time (PDT),West,0.995050,8.1875,202
64,Hialeah,Florida,Suburb,Eastern Daylight Time (EDT),South,0.984756,8.56044,328
144,San Jose,California,City,Pacific Daylight Time (PDT),West,0.983051,7.428571,59
19,Calumet City,Illinois,Suburb,Central Daylight Time (CDT),Midwest,0.981481,9.025641,54
127,Phoenix,Arizona,Suburb,Mountain Daylight Time (MDT),West,0.978261,8.92,46
66,Homestead,Florida,Suburb,Eastern Daylight Time (EDT),South,0.975610,7.867925,246
161,Warren,Rhode Island,Rural,Eastern Daylight Time (EDT),Northeast,0.975610,8.571429,41


In [3]:
!pip install plotly
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


# Load data
students = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_Student(Seat)')
high_schools = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_High_School')
survey = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='SurveyData_Student(Seat)')

# Filter valid records
students = students[students["Participation Status"] == "Participating"]
students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)

# Merge school + NPS data
merged = students.merge(survey[["Seat Id", "NPS"]], on="Seat Id", how="left")
merged = merged.merge(high_schools[["School Id", "State"]], on="School Id", how="left")

# Region map using full state names
def get_region(state):
    northeast = ["New York", "New Jersey", "Massachusetts", "Pennsylvania", "Connecticut",
                 "Rhode Island", "Vermont", "New Hampshire", "Maine"]
    midwest = ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin",
               "Minnesota", "Iowa", "Missouri", "Kansas", "Nebraska", "South Dakota", "North Dakota"]
    south = ["Texas", "Florida", "Georgia", "North Carolina", "South Carolina", "Virginia",
             "Alabama", "Mississippi", "Kentucky", "Tennessee", "Oklahoma", "Arkansas",
             "Louisiana", "West Virginia", "Delaware", "Maryland", "District of Columbia"]
    west = ["California", "Arizona", "Colorado", "Washington", "Oregon", "Nevada",
            "Utah", "New Mexico", "Hawaii", "Alaska", "Idaho", "Montana", "Wyoming"]

    if state in northeast:
        return "Northeast"
    elif state in midwest:
        return "Midwest"
    elif state in south:
        return "South"
    elif state in west:
        return "West"
    else:
        return "Other"

merged["Region"] = merged["State"].apply(get_region)

# Aggregate per student
student_summary = merged.groupby(["Student Id", "Region"]).agg(
    Courses_Taken=("Seat Id", "count"),
    Pass_Rate=("Passed_Flag", "mean"),
    Avg_NPS=("NPS", "mean")
).reset_index()

# Normalize NPS to 0–1 scale, compute balanced score
student_summary["Satisfaction_Score"] = student_summary["Avg_NPS"] / 10
student_summary["Balanced_Score"] = 0.5 * student_summary["Pass_Rate"] + 0.5 * student_summary["Satisfaction_Score"]


<ipython-input-3-fb0037e31e6c>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)


In [17]:
fig = px.scatter(
    student_summary[student_summary["Region"] == "Northeast"],
    x="Courses_Taken",
    y="Balanced_Score",
    labels={
        "Courses_Taken": "Number of College Courses Taken",
        "Balanced_Score": "Student Success Score (Pass Rate + Satisfaction)"
    },
    title=" (Northeast Region) Course Load v Student Success ",
    template="simple_white"
)

fig.update_traces(marker=dict(color='crimson', size=8, opacity=0.7))
fig.update_layout(title_font_size=20)
fig.show()


In [18]:
fig = px.scatter(
    student_summary[student_summary["Region"] == "Midwest"],
    x="Courses_Taken",
    y="Balanced_Score",
    labels={
        "Courses_Taken": "Number of College Courses Taken",
        "Balanced_Score": "Student Success Score (Pass Rate + Satisfaction)"
    },
    title=" (Midwest Region) Course Load v Student Success ",
    template="simple_white"
)

fig.update_traces(marker=dict(color='green', size=8, opacity=0.7))
fig.update_layout(title_font_size=20)
fig.show()



In [19]:
fig = px.scatter(
    student_summary[student_summary["Region"] == "South"],
    x="Courses_Taken",
    y="Balanced_Score",
    labels={
        "Courses_Taken": "Number of College Courses Taken",
        "Balanced_Score": "Student Success Score (Pass Rate + Satisfaction)"
    },
    title=" (South Region) Course Load v Student Success ",
    template="simple_white"
)

fig.update_traces(marker=dict(color='blue', size=8, opacity=0.7))
fig.update_layout(title_font_size=20)
fig.show()



In [20]:
fig = px.scatter(
    student_summary[student_summary["Region"] == "West"],
    x="Courses_Taken",
    y="Balanced_Score",
    labels={
        "Courses_Taken": "Number of College Courses Taken",
        "Balanced_Score": "Student Success Score (Pass Rate + Satisfaction)"
    },
    title=" (West Region) Course Load v Student Success ",
    template="simple_white"
)

fig.update_traces(marker=dict(color='orange', size=8, opacity=0.7))
fig.update_layout(title_font_size=20)
fig.show()



In [8]:
fig = px.scatter(
    student_summary,
    x="Courses_Taken",
    y="Balanced_Score",
    color="Region",  # Color by region
    labels={
        "Courses_Taken": "Number of Courses",
        "Balanced_Score": "Balanced Score (50% Pass + 50% NPS)"
    },
    title="All Regions: Courses Taken vs Balanced Score",
    template="simple_white"
)

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.show()

In [9]:
heatmap_data = student_summary.groupby(["Region", "Courses_Taken"]).agg(
    Avg_Balanced_Score=("Balanced_Score", "mean")
).reset_index()

fig = px.density_heatmap(
    heatmap_data,
    x="Courses_Taken",
    y="Region",
    z="Avg_Balanced_Score",
    color_continuous_scale="Viridis",
    title="Average Balanced Score by Courses Taken and Region",
    template="simple_white"
)

fig.show()

In [10]:
fig = px.box(
    student_summary,
    x="Region",
    y="Balanced_Score",
    points="all",  # Shows raw points too
    title="Distribution of Balanced Success Scores by Region",
    labels={"Balanced_Score": "Balanced Score (50% Pass + 50% NPS)"},
    template="simple_white"
)

fig.update_traces(marker=dict(opacity=0.5))
fig.show()


In [23]:
import pandas as pd
import plotly.express as px

# Load your student and school datasets
students = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_Student(Seat)')
high_schools = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_High_School')
survey = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='SurveyData_Student(Seat)')

# Filter and clean student data
students = students[students["Participation Status"] == "Participating"]
students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)

# Merge student data with survey and school info
merged = students.merge(survey[["Seat Id", "NPS"]], on="Seat Id", how="left")
merged = merged.merge(high_schools, left_on="School Id", right_on="School Id", how="left")

# Assign U.S. Census Regions
def get_region(state):
    northeast = ["New York", "New Jersey", "Massachusetts", "Pennsylvania", "Connecticut",
                 "Rhode Island", "Vermont", "New Hampshire", "Maine"]
    midwest = ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin",
               "Minnesota", "Iowa", "Missouri", "Kansas", "Nebraska", "South Dakota", "North Dakota"]
    south = ["Texas", "Florida", "Georgia", "North Carolina", "South Carolina", "Virginia",
             "Alabama", "Mississippi", "Kentucky", "Tennessee", "Oklahoma", "Arkansas",
             "Louisiana", "West Virginia", "Delaware", "Maryland", "District of Columbia"]
    west = ["California", "Arizona", "Colorado", "Washington", "Oregon", "Nevada",
            "Utah", "New Mexico", "Hawaii", "Alaska", "Idaho", "Montana", "Wyoming"]

    if state in northeast:
        return "Northeast"
    elif state in midwest:
        return "Midwest"
    elif state in south:
        return "South"
    elif state in west:
        return "West"
    else:
        return "Other"

merged["Region"] = merged["State"].apply(get_region)

# Calculate student success score: 50% pass + 50% satisfaction
merged["Success_Score"] = 0.5 * merged["Passed_Flag"] + 0.5 * (merged["NPS"] / 10)

# Step 1: For each student, count courses and average success
student_summary = merged.groupby(["Region", "Locale", "Student Id"]).agg(
    Courses_Taken=("Seat Id", "count"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

summary = summary.dropna(subset=["Region", "Locale", "Courses_Taken", "Mean_Success_Score"])

# Step 2: For each Region + Locale + Course Count, get average success
summary = student_summary.groupby(["Region", "Locale", "Courses_Taken"]).agg(
    Mean_Success_Score=("Avg_Success_Score", "mean"),
    Student_Count=("Student Id", "count")
).reset_index()

# Step 3: Only keep groupings with at least 10 students
summary = summary[summary["Student_Count"] >= 10]

# 🔧 Drop any rows with NaNs in key grouping columns
summary = summary.dropna(subset=["Region", "Locale", "Courses_Taken", "Mean_Success_Score"])

# Step 4: For each Region + Locale, find the course count with the highest success
optimal_idx = summary.groupby(["Region", "Locale"])["Mean_Success_Score"].idxmax()
optimal = summary.loc[optimal_idx]


# Step 5: Label for chart
optimal["Region_Locale"] = optimal["Region"] + " - " + optimal["Locale"]

# Step 6: Plot grouped bar chart
fig = px.bar(
    optimal,
    x="Region_Locale",
    y="Mean_Success_Score",
    color="Region",
    text="Courses_Taken",
    title="Optimal Number of Courses for Maximum Student Success by Region and Locale",
    labels={
        "Mean_Success_Score": "Avg Student Success Score",
        "Region_Locale": "Region + Locale"
    },
    template="simple_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(xaxis_tickangle=-45, title_font_size=18)
fig.show()


<ipython-input-23-0282287ee23e>:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [24]:
import pandas as pd
import plotly.express as px

# Load datasets
students = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_Student(Seat)')
high_schools = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='AdminData_High_School')
survey = pd.read_excel('EdEquityLab_Dataset_DataforGood_040325.xlsx', sheet_name='SurveyData_Student(Seat)')

# Clean and prep
students = students[students["Participation Status"] == "Participating"]
students["Passed_Flag"] = students["Pass Status"].apply(lambda x: 1 if x == "Passed" else 0)

# Merge
merged = students.merge(survey[["Seat Id", "NPS"]], on="Seat Id", how="left")
merged = merged.merge(high_schools, on="School Id", how="left")

# Region assign
def get_region(state):
    northeast = ["New York", "New Jersey", "Massachusetts", "Pennsylvania", "Connecticut",
                 "Rhode Island", "Vermont", "New Hampshire", "Maine"]
    midwest = ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin",
               "Minnesota", "Iowa", "Missouri", "Kansas", "Nebraska", "South Dakota", "North Dakota"]
    south = ["Texas", "Florida", "Georgia", "North Carolina", "South Carolina", "Virginia",
             "Alabama", "Mississippi", "Kentucky", "Tennessee", "Oklahoma", "Arkansas",
             "Louisiana", "West Virginia", "Delaware", "Maryland", "District of Columbia"]
    west = ["California", "Arizona", "Colorado", "Washington", "Oregon", "Nevada",
            "Utah", "New Mexico", "Hawaii", "Alaska", "Idaho", "Montana", "Wyoming"]

    if state in northeast:
        return "Northeast"
    elif state in midwest:
        return "Midwest"
    elif state in south:
        return "South"
    elif state in west:
        return "West"
    else:
        return "Other"

merged["Region"] = merged["State"].apply(get_region)
merged["Success_Score"] = 0.5 * merged["Passed_Flag"] + 0.5 * (merged["NPS"] / 10)

# Group per student to count courses and success
student_summary = merged.groupby(["Region", "Locale", "Student Id"]).agg(
    Courses_Taken=("Seat Id", "count"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

# Group by Region + Locale + Courses_Taken
course_outcomes = student_summary.groupby(["Region", "Locale", "Courses_Taken"]).agg(
    Mean_Success_Score=("Avg_Success_Score", "mean"),
    Student_Count=("Student Id", "count")
).reset_index()

# Drop NaNs & filter for solid samples
course_outcomes = course_outcomes.dropna(subset=["Region", "Locale", "Courses_Taken", "Mean_Success_Score"])
course_outcomes = course_outcomes[course_outcomes["Student_Count"] >= 10]

# Find optimal number of courses (highest success) for each Region + Locale
optimal = course_outcomes.loc[course_outcomes.groupby(["Region", "Locale"])["Mean_Success_Score"].idxmax()]

# Also calculate average optimal course count across all locales for each Region
region_avg = student_summary.groupby(["Region", "Courses_Taken"]).agg(
    Mean_Success_Score=("Avg_Success_Score", "mean"),
    Student_Count=("Student Id", "count")
).reset_index()

region_avg = region_avg[region_avg["Student_Count"] >= 10]
region_best = region_avg.loc[region_avg.groupby("Region")["Mean_Success_Score"].idxmax()]
region_best["Locale"] = "Average"

# Combine with optimal suburb/rural/etc.
final_df = pd.concat([optimal, region_best], ignore_index=True)
final_df = final_df[final_df["Region"].isin(["Northeast", "Midwest", "South", "West"])]

# Plot grouped bar chart
fig = px.bar(
    final_df,
    x="Region",
    y="Courses_Taken",
    color="Locale",
    barmode="group",
    text="Courses_Taken",
    title="Optimal Number of Courses by Region and Locale",
    labels={"Courses_Taken": "Optimal Number of Courses", "Region": "U.S. Region"},
    template="simple_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(title_font_size=20)
fig.show()


<ipython-input-24-c4846c8d9ebc>:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

